In [1]:
# =====================================================================
#  Optimized Hybrid Models – Imports & Configuration
# =====================================================================
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
import time

# ---- Paths ----------------------------------------------------------
CONFIG = {
    "DATA_ROOT":       "./prepareddata",   # raw val/test CSVs (with target)
    "TRAINED_ROOT":    "./trained",        # saved model artifacts
    "LEADERBOARD_DIR": "./trained/leaderboards",
    "OUTPUT_DIR":      "./trained/optimized_hybrid",
    "TARGET_COL":      "Fraud",
    "TOP_N_MODELS":    10,                 # candidates from leaderboard
    "MAX_MODELS":      5,                  # maximum number of models to select
    "RANDOM_STATE":    42,
}

Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
print(f"[config] DATA_ROOT      = {CONFIG['DATA_ROOT']}")
print(f"[config] TRAINED_ROOT   = {CONFIG['TRAINED_ROOT']}")
print(f"[config] OUTPUT_DIR     = {CONFIG['OUTPUT_DIR']}")

[config] DATA_ROOT      = ./prepareddata
[config] TRAINED_ROOT   = ./trained
[config] OUTPUT_DIR     = ./trained/optimized_hybrid


In [2]:
# =====================================================================
#  Helper Functions
# =====================================================================
def compute_metrics(y_true, y_proba, threshold=0.5):
    """Compute fraud-detection metrics (positive label = 1)."""
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "auc_roc":   roc_auc_score(y_true, y_proba),
        "auc_pr":    average_precision_score(y_true, y_proba),
        "n_pos":     int(np.sum(y_true)),
        "n_total":   int(len(y_true)),
    }


def load_raw_labels(dataset_name, split="val"):
    """Load true labels from the raw split CSV (contains target column)."""
    path = Path(CONFIG["DATA_ROOT"]) / f"{dataset_name}_{split}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing raw {split} file: {path}")
    df = pd.read_csv(path)
    if CONFIG["TARGET_COL"] not in df.columns:
        raise ValueError(f"Target column '{CONFIG['TARGET_COL']}' not found in {path}")
    return df[CONFIG["TARGET_COL"]].values.astype(int)


def load_model_predictions(dataset_name, variant, model, split="val"):
    """
    Load the probability column for a specific base model from
    the saved meta-features CSV of a given variant.
    """
    meta_path = Path(CONFIG["TRAINED_ROOT"]) / dataset_name / variant / f"{split}_meta_features.csv"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing meta-features file: {meta_path}")
    meta_df = pd.read_csv(meta_path)
    if model not in meta_df.columns:
        raise ValueError(f"Model '{model}' not found in {meta_path}. Available: {list(meta_df.columns)}")
    return meta_df[model].values

def get_diverse_candidates(dataset_name):
    """
    Read top‑10 L0 leaderboard and return exactly one candidate per
    unique model type, choosing the variant with the highest F1.
    """
    lb_path = Path(CONFIG["LEADERBOARD_DIR"]) / f"{dataset_name}_top10_L0_by_f1.csv"
    if not lb_path.exists():
        raise FileNotFoundError(f"Leaderboard not found: {lb_path}")

    lb = pd.read_csv(lb_path)
    # Keep only the best variant for each model type
    best_per_model = lb.sort_values(by="f1", ascending=False).drop_duplicates(
        subset=["model"], keep="first"
    )

    candidates = []
    for _, row in best_per_model.iterrows():
        candidates.append({
            "variant": row["variant"],
            "model":   row["model"],
            "f1_leaderboard": row["f1"],
        })

    print(f"[diverse candidates] {dataset_name}: {len(candidates)} unique model types")
    return candidates

def cv_evaluate_subset(
    dataset_name, subset_keys, meta_learner, y_val, n_splits=3, random_state=42
):
    """
    Evaluate a subset of models with a given meta‑learner using
    stratified cross‑validation on the validation set.
    Returns the average F1 over folds.
    """
    X_val_meta = build_meta_features(subset_keys, dataset_name, split="val")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_f1s = []

    for tr_idx, va_idx in skf.split(X_val_meta, y_val):
        X_tr, X_va = X_val_meta.iloc[tr_idx], X_val_meta.iloc[va_idx]
        y_tr, y_va = y_val[tr_idx], y_val[va_idx]

        ml = clone(meta_learner)
        try:
            ml.fit(X_tr, y_tr)
            if hasattr(ml, "predict_proba"):
                proba = ml.predict_proba(X_va)[:, 1]
            elif hasattr(ml, "decision_function"):
                from scipy.special import expit
                proba = expit(ml.decision_function(X_va))
            else:
                continue
            pred = (proba >= 0.5).astype(int)
            f1 = f1_score(y_va, pred, zero_division=0)
            fold_f1s.append(f1)
        except Exception as e:
            print(f"    [cv] meta-learner failed: {e}")
            continue

    if not fold_f1s:
        return -np.inf
    return float(np.mean(fold_f1s))

In [3]:
# =====================================================================
#  Build Candidate Pool from Saved Leaderboards
# =====================================================================
def get_candidates(dataset_name):
    """Read top-N single models from the leaderboard CSV."""
    lb_path = Path(CONFIG["LEADERBOARD_DIR"]) / f"{dataset_name}_top10_L0_by_f1.csv"
    if not lb_path.exists():
        # fallback to top3? maybe old naming
        lb_path = Path(CONFIG["LEADERBOARD_DIR"]) / f"{dataset_name}_top10_L0_by_f1.csv"
    if not lb_path.exists():
        raise FileNotFoundError(f"Leaderboard not found: {lb_path}")

    lb = pd.read_csv(lb_path)
    # Ensure sorted by F1 descending
    lb = lb.sort_values(by="f1", ascending=False).head(CONFIG["TOP_N_MODELS"])
    candidates = []
    for _, row in lb.iterrows():
        candidates.append({
            "variant": row["variant"],
            "model":   row["model"],
            "f1_leaderboard": row["f1"],
        })
    print(f"[candidates] {dataset_name}: {len(candidates)} candidates")
    return candidates

In [4]:
# =====================================================================
#  Build Candidate Pool from ALL Trained Models (One per Model Type)
# =====================================================================
def get_best_per_model_all(dataset_name):
    """
    Scan every variant directory for this dataset, compute test F1 for each
    base model column, and return the best (variant, model) pair for each
    distinct model type.
    """
    trained_root = Path(CONFIG["TRAINED_ROOT"]) / dataset_name
    if not trained_root.exists():
        raise FileNotFoundError(f"No trained models directory: {trained_root}")

    # Store best record per model type
    best_records = {}   # model -> (variant, f1)

    # Iterate over all variant subdirectories
    for variant_dir in trained_root.iterdir():
        if not variant_dir.is_dir():
            continue
        meta_path = variant_dir / "test_meta_features.csv"
        pred_path = variant_dir / "test_predictions.csv"
        if not meta_path.exists() or not pred_path.exists():
            continue

        # Load true labels
        pred_df = pd.read_csv(pred_path)
        if "y_true" not in pred_df.columns:
            continue
        y_true = pred_df["y_true"].values.astype(int)

        # Load meta features (one column per base model)
        meta_df = pd.read_csv(meta_path)

        # Compute F1 for each model column
        for model_col in meta_df.columns:
            proba = meta_df[model_col].values
            if np.any(np.isnan(proba)):
                proba = np.nan_to_num(proba, nan=0.5)
            y_pred = (proba >= 0.5).astype(int)
            try:
                f1 = f1_score(y_true, y_pred, zero_division=0)
            except Exception:
                continue

            # Update best for this model type if improved
            if model_col not in best_records or f1 > best_records[model_col][1]:
                best_records[model_col] = (variant_dir.name, f1)

    if not best_records:
        raise RuntimeError(f"No base model results found for dataset {dataset_name}")

    # Convert to candidate list
    candidates = []
    for model, (variant, f1) in best_records.items():
        candidates.append({
            "variant": variant,
            "model": model,
            "f1_leaderboard": f1,
        })

    print(f"[candidates all] {dataset_name}: {len(candidates)} unique model types")
    return candidates

In [5]:
# =====================================================================
#  Greedy Max Coverage Selection (Fraud Instances Only)
# =====================================================================
def greedy_select_models(candidates, y_val, dataset_name):
    """
    Select a small subset of candidates that collectively cover the
    maximum number of fraud validation instances correctly.
    """
    n_fraud = int(np.sum(y_val == 1))
    if n_fraud == 0:
        raise ValueError("No fraud instances in validation set!")

    # Precompute correct fraud coverage for each candidate
    coverage_masks = {}
    candidate_info = {}
    for cand in candidates:
        try:
            proba = load_model_predictions(
                dataset_name, cand["variant"], cand["model"], split="val"
            )
            pred = (proba >= 0.5).astype(int)
            fraud_mask = (y_val == 1)
            correct_fraud = (pred == y_val) & fraud_mask   # true positives only
            coverage_masks[(cand["variant"], cand["model"])] = correct_fraud
            candidate_info[(cand["variant"], cand["model"])] = cand
        except Exception as e:
            print(f"  [skip] {cand['variant']} / {cand['model']}: {e}")

    if not coverage_masks:
        raise RuntimeError("No candidate models could be loaded.")

    selected = []
    union_mask = np.zeros(len(y_val), dtype=bool)

    while len(selected) < CONFIG["MAX_MODELS"]:
        best_key = None
        best_gain = 0
        best_extra_mask = None

        for key, mask in coverage_masks.items():
            if key in selected:
                continue
            new_mask = mask & ~union_mask
            gain = new_mask.sum()
            if gain > best_gain:
                best_gain = gain
                best_key = key
                best_extra_mask = new_mask
            elif gain == best_gain and best_key is not None:
                # tie-break: prefer higher leaderboard F1
                curr_f1 = candidate_info[best_key]["f1_leaderboard"]
                cand_f1 = candidate_info[key]["f1_leaderboard"]
                if cand_f1 > curr_f1:
                    best_key = key
                    best_extra_mask = new_mask

        if best_key is None or best_gain == 0:
            break

        selected.append(best_key)
        union_mask |= best_extra_mask
        print(f"  [select] {best_key[0][:40]:40s} | {best_key[1]:8s} "
              f"new fraud covered: {best_gain:3d} | total covered: {union_mask.sum():3d}/{n_fraud}")

    coverage_rate = union_mask.sum() / n_fraud
    print(f"  [greedy] selected {len(selected)} models, "
          f"fraud coverage = {coverage_rate:.4f} ({union_mask.sum()}/{n_fraud})")
    return selected, coverage_rate

In [6]:
# =====================================================================
#  Balanced Greedy Selection (Fraud Coverage vs Normal False Positives)
# =====================================================================
def balanced_greedy_select_models(
    candidates, y_val, dataset_name, lambda_fp=0.5, max_models=None
):
    """
    Select a subset of models that covers frauds well **while penalizing
    false positives** on normal transactions.

    At each step, choose the model with the highest:
        (new fraud correctly covered) - lambda_fp * (new normal misclassified as fraud)
    """
    if max_models is None:
        max_models = CONFIG["MAX_MODELS"]

    n_fraud = int(np.sum(y_val == 1))
    if n_fraud == 0:
        raise ValueError("No fraud instances in validation set!")

    # Precompute coverage masks (true positive fraud) and false‑positive masks (normal predicted fraud)
    coverage_masks = {}
    fp_masks = {}
    candidate_info = {}

    for cand in candidates:
        try:
            proba = load_model_predictions(
                dataset_name, cand["variant"], cand["model"], split="val"
            )
            pred = (proba >= 0.5).astype(int)
            fraud_mask = (y_val == 1)
            normal_mask = (y_val == 0)

            correct_fraud = (pred == y_val) & fraud_mask   # true positives
            false_pos = (pred == 1) & normal_mask          # predicted fraud but actually normal

            key = (cand["variant"], cand["model"])
            coverage_masks[key] = correct_fraud
            fp_masks[key] = false_pos
            candidate_info[key] = cand
        except Exception as e:
            print(f"  [skip] {cand['variant']} / {cand['model']}: {e}")

    if not coverage_masks:
        raise RuntimeError("No candidate models could be loaded.")

    selected = []
    union_tp = np.zeros(len(y_val), dtype=bool)   # frauds already covered
    union_fp = np.zeros(len(y_val), dtype=bool)   # normals already falsely flagged

    while len(selected) < max_models:
        best_key = None
        best_score = -np.inf
        best_new_tp = None
        best_new_fp = None

        for key in coverage_masks.keys():
            if key in selected:
                continue

            new_tp = coverage_masks[key] & ~union_tp
            new_fp = fp_masks[key] & ~union_fp

            gain_tp = new_tp.sum()
            gain_fp = new_fp.sum()

            # Weighted score: reward fraud coverage, penalise normal false positives
            score = gain_tp - lambda_fp * gain_fp

            if score > best_score:
                best_score = score
                best_key = key
                best_new_tp = new_tp
                best_new_fp = new_fp
            elif score == best_score and best_key is not None:
                # tie-break: prefer higher leaderboard F1
                curr_f1 = candidate_info[best_key]["f1_leaderboard"]
                cand_f1 = candidate_info[key]["f1_leaderboard"]
                if cand_f1 > curr_f1:
                    best_key = key
                    best_new_tp = new_tp
                    best_new_fp = new_fp

        if best_key is None or best_score <= 0:
            # stop if no model improves the score
            break

        selected.append(best_key)
        union_tp |= best_new_tp
        union_fp |= best_new_fp

        print(f"  [balanced] {best_key[0][:40]:40s} | {best_key[1]:8s} "
              f"score={best_score:6.2f} (new TP={best_new_tp.sum():3d}, "
              f"new FP={best_new_fp.sum():3d})")

    coverage_rate = union_tp.sum() / n_fraud
    false_alarm_count = union_fp.sum()
    print(f"  [balanced] selected {len(selected)} models, "
          f"fraud coverage = {coverage_rate:.4f} ({union_tp.sum()}/{n_fraud}), "
          f"false alarms = {false_alarm_count}")
    return selected, coverage_rate, false_alarm_count

In [7]:
# =====================================================================
#  Exhaustive Subset Search with F1‑based Early Stopping
# =====================================================================


def exhaustive_subset_search(
    candidates, y_val, dataset_name, max_models=None,
    coverage_weight=1.0, size_penalty=0.01, patience=2
):
    """
    Exhaustively try all combinations of models of size 2, 3, ... up to
    max_models. For each size, the best subset is chosen by coverage score.
    Then a Logistic Regression meta‑learner is trained on validation
    meta‑features and its F1 is computed.
    The search stops if validation F1 does not improve for `patience`
    consecutive size increments. The subset with the highest validation F1
    is returned.

    Parameters:
      coverage_weight : importance of fraud coverage in the fast scoring
      size_penalty    : penalty per extra model (encourages parsimony)
      patience        : number of consecutive non‑improvements allowed
    """
    if max_models is None:
        max_models = CONFIG["MAX_MODELS"]

    n_fraud = int(np.sum(y_val == 1))
    if n_fraud == 0:
        raise ValueError("No fraud instances in validation set!")

    # Precompute binary correct‑fraud masks for each candidate
    model_keys = []
    coverage_masks = {}
    candidate_info = {}

    for cand in candidates:
        try:
            proba = load_model_predictions(
                dataset_name, cand["variant"], cand["model"], split="val"
            )
            pred = (proba >= 0.5).astype(int)
            fraud_mask = (y_val == 1)
            correct_fraud = (pred == y_val) & fraud_mask
            key = (cand["variant"], cand["model"])
            model_keys.append(key)
            coverage_masks[key] = correct_fraud
            candidate_info[key] = cand
        except Exception as e:
            print(f"  [skip] {cand['variant']} / {cand['model']}: {e}")

    if len(model_keys) < 2:
        print("  [exhaustive] Fewer than 2 candidates available, falling back to best single model")
        best_key = max(candidate_info.items(), key=lambda kv: kv[1]["f1_leaderboard"])[0]
        return [best_key], coverage_masks[best_key].sum() / n_fraud, 0.0

    best_overall_subset = None
    best_overall_f1 = -np.inf
    best_overall_coverage = 0.0
    best_overall_size = 0

    # Early stopping variables
    no_improve_count = 0
    prev_best_f1 = -np.inf

    for size in range(2, max_models + 1):
        best_subset_this_size = None
        best_score_this_size = -np.inf
        best_coverage_this_size = 0.0

        # Enumerate all combinations of this size
        for subset in combinations(model_keys, size):
            union_mask = np.zeros(len(y_val), dtype=bool)
            for key in subset:
                union_mask |= coverage_masks[key]
            coverage = union_mask.sum() / n_fraud
            score = coverage_weight * coverage - size_penalty * len(subset)

            if score > best_score_this_size:
                best_score_this_size = score
                best_subset_this_size = subset
                best_coverage_this_size = coverage

        # --- Evaluate the best subset of this size on validation -----
        try:
            # Build validation meta‑features for the subset
            meta_val = build_meta_features(list(best_subset_this_size), dataset_name, split="val")
            # Use a simple, fast meta‑learner to avoid heavy computation
            meta_clf = LogisticRegression(C=1.0, max_iter=500, random_state=CONFIG["RANDOM_STATE"])
            meta_clf.fit(meta_val, y_val)
            val_proba = meta_clf.predict_proba(meta_val)[:, 1]
            val_pred = (val_proba >= 0.5).astype(int)
            val_f1 = f1_score(y_val, val_pred, zero_division=0)
        except Exception as e:
            print(f"  [exhaustive] size={size}: evaluation failed: {e}")
            val_f1 = -np.inf

        print(f"  [exhaustive] size={size}: coverage={best_coverage_this_size:.4f}, "
              f"validation F1={val_f1:.4f}")

        # --- Update overall best if improved --------------------------
        if val_f1 > best_overall_f1:
            best_overall_f1 = val_f1
            best_overall_subset = best_subset_this_size
            best_overall_coverage = best_coverage_this_size
            best_overall_size = size
            no_improve_count = 0
        else:
            no_improve_count += 1

        # --- Early stopping: stop if no improvement for `patience` sizes
        if no_improve_count >= patience:
            print(f"  [exhaustive] stopping early: validation F1 did not improve "
                  f"for {patience} consecutive size increments.")
            break

    if best_overall_subset is None:
        # Fallback to best single model
        print("  [exhaustive] No valid subset found, falling back to best single model")
        best_key = max(candidate_info.items(), key=lambda kv: kv[1]["f1_leaderboard"])[0]
        best_overall_subset = (best_key,)
        best_overall_coverage = coverage_masks[best_key].sum() / n_fraud
        best_overall_size = 1
        # F1 unknown; set to 0
        best_overall_f1 = 0.0

    print(f"  [exhaustive] selected {best_overall_size} models, "
          f"validation F1={best_overall_f1:.4f}, fraud coverage={best_overall_coverage:.4f}")
    return list(best_overall_subset), best_overall_coverage, best_overall_f1

In [8]:
# =====================================================================
#  Heuristic Beam Search + Cross‑Validated F1
# =====================================================================

def heuristic_cv_subset_search(
    candidates, y_val, dataset_name, max_models=None,
    meta_learners=None, n_splits=3, patience=2
):
    """
    Heuristic beam search over subsets of diverse candidates.

    - size=2: try all pairs
    - size>=3: take the best subset from previous size and add each remaining candidate
    - evaluation: for each subset, try all meta‑learners with CV F1
    - early stopping if CV F1 does not improve for `patience` sizes

    Returns:
      selected_keys, best_meta_name, best_cv_f1, coverage
    """
    if max_models is None:
        max_models = CONFIG["MAX_MODELS"]
    if meta_learners is None:
        meta_learners = get_meta_learners(CONFIG["RANDOM_STATE"])

    model_keys = [(c["variant"], c["model"]) for c in candidates]
    n_candidates = len(model_keys)
    if n_candidates < 2:
        raise RuntimeError("Need at least 2 candidates for subset search.")

    best_overall_subset = None
    best_overall_meta = None
    best_overall_cv_f1 = -np.inf
    best_overall_size = 0

    no_improve_count = 0
    prev_best_cv_f1 = -np.inf

    # ---------------- size = 2: all pairs ----------------
    best_pair = None
    best_pair_meta = None
    best_pair_f1 = -np.inf

    for pair in combinations(model_keys, 2):
        for meta_name, meta_learner in meta_learners.items():
            cv_f1 = cv_evaluate_subset(
                dataset_name, list(pair), meta_learner, y_val,
                n_splits=n_splits, random_state=CONFIG["RANDOM_STATE"]
            )
            if cv_f1 > best_pair_f1:
                best_pair_f1 = cv_f1
                best_pair = pair
                best_pair_meta = meta_name

    print(f"  [heur] size=2: best CV F1={best_pair_f1:.4f} (meta={best_pair_meta})")
    best_overall_subset = best_pair
    best_overall_meta = best_pair_meta
    best_overall_cv_f1 = best_pair_f1
    best_overall_size = 2
    prev_best_cv_f1 = best_pair_f1

    # ---------------- size >= 3: beam expansion ----------------
    current_best_subset = list(best_pair)
    current_best_meta = best_pair_meta
    current_best_f1 = best_pair_f1

    for size in range(3, min(max_models, n_candidates) + 1):
        # Generate candidate subsets: current best + each remaining model
        remaining = [k for k in model_keys if k not in current_best_subset]
        if not remaining:
            break

        best_candidate_subset = None
        best_candidate_meta = None
        best_candidate_f1 = -np.inf

        for extra_key in remaining:
            candidate_subset = current_best_subset + [extra_key]
            for meta_name, meta_learner in meta_learners.items():
                cv_f1 = cv_evaluate_subset(
                    dataset_name, candidate_subset, meta_learner, y_val,
                    n_splits=n_splits, random_state=CONFIG["RANDOM_STATE"]
                )
                if cv_f1 > best_candidate_f1:
                    best_candidate_f1 = cv_f1
                    best_candidate_subset = candidate_subset
                    best_candidate_meta = meta_name

        print(f"  [heur] size={size}: best CV F1={best_candidate_f1:.4f} (meta={best_candidate_meta})")

        # Update overall best
        if best_candidate_f1 > best_overall_cv_f1:
            best_overall_cv_f1 = best_candidate_f1
            best_overall_subset = best_candidate_subset
            best_overall_meta = best_candidate_meta
            best_overall_size = size
            no_improve_count = 0
        else:
            no_improve_count += 1

        # Move to next size
        current_best_subset = best_candidate_subset
        current_best_meta = best_candidate_meta
        current_best_f1 = best_candidate_f1
        prev_best_cv_f1 = best_candidate_f1

        if no_improve_count >= patience:
            print(f"  [heur] early stopping: CV F1 did not improve for {patience} sizes.")
            break

    # Compute fraud coverage on validation
    union_mask = np.zeros(len(y_val), dtype=bool)
    for key in best_overall_subset:
        proba = load_model_predictions(dataset_name, key[0], key[1], split="val")
        pred = (proba >= 0.5).astype(int)
        fraud_mask = (y_val == 1)
        union_mask |= (pred == y_val) & fraud_mask
    coverage = union_mask.sum() / int(np.sum(y_val == 1))

    print(f"  [heur] selected {best_overall_size} models, meta-learner={best_overall_meta}, "
          f"CV F1={best_overall_cv_f1:.4f}, fraud coverage={coverage:.4f}")

    return list(best_overall_subset), best_overall_meta, best_overall_cv_f1, coverage

In [9]:
# =====================================================================
#  Balanced Coverage Greedy (Fraud + Genuine)
# =====================================================================
def balanced_coverage_greedy(
    candidates, y_val, dataset_name, max_models=None
):
    """
    Greedily select models that maximise:
        (fraud_coverage + genuine_coverage) / 2
    on the validation set. Stops when score does not improve or max_models reached.

    Returns:
      selected_keys, best_score, coverage_fraud, coverage_genuine
    """
    if max_models is None:
        max_models = CONFIG["MAX_MODELS"]

    n_total = len(y_val)
    n_fraud = int(np.sum(y_val == 1))
    n_genuine = n_total - n_fraud
    if n_fraud == 0 or n_genuine == 0:
        raise ValueError("Validation set must contain both fraud and genuine samples.")

    # Precompute masks for each candidate
    coverage_fraud_masks = {}
    coverage_genuine_masks = {}
    candidate_info = {}

    for cand in candidates:
        try:
            proba = load_model_predictions(
                dataset_name, cand["variant"], cand["model"], split="val"
            )
            pred = (proba >= 0.5).astype(int)
            fraud_mask = (y_val == 1)
            genuine_mask = (y_val == 0)

            correct_fraud = (pred == y_val) & fraud_mask
            correct_genuine = (pred == y_val) & genuine_mask

            key = (cand["variant"], cand["model"])
            coverage_fraud_masks[key] = correct_fraud
            coverage_genuine_masks[key] = correct_genuine
            candidate_info[key] = cand
        except Exception as e:
            print(f"  [skip] {cand['variant']} / {cand['model']}: {e}")

    if not coverage_fraud_masks:
        raise RuntimeError("No candidate models could be loaded.")

    selected = []
    union_fraud = np.zeros(n_total, dtype=bool)
    union_genuine = np.zeros(n_total, dtype=bool)

    best_score = -1.0
    best_selection = []
    best_union_fraud = None
    best_union_genuine = None

    while len(selected) < max_models:
        best_key = None
        best_new_score = -1.0
        best_new_fraud = None
        best_new_genuine = None

        for key in coverage_fraud_masks.keys():
            if key in selected:
                continue

            new_fraud = coverage_fraud_masks[key] & ~union_fraud
            new_genuine = coverage_genuine_masks[key] & ~union_genuine

            new_fraud_rate = new_fraud.sum() / n_fraud
            new_genuine_rate = new_genuine.sum() / n_genuine
            incremental_score = (new_fraud_rate + new_genuine_rate) / 2.0

            # Total score after adding this model
            total_fraud = union_fraud | new_fraud
            total_genuine = union_genuine | new_genuine
            total_score = (total_fraud.sum()/n_fraud + total_genuine.sum()/n_genuine) / 2.0

            # We select based on incremental improvement, but ensure total score is non-decreasing
            if incremental_score > best_new_score:
                best_new_score = incremental_score
                best_key = key
                best_new_fraud = new_fraud
                best_new_genuine = new_genuine

        if best_key is None or best_new_score <= 0:
            break

        selected.append(best_key)
        union_fraud |= best_new_fraud
        union_genuine |= best_new_genuine

        total_score = (union_fraud.sum()/n_fraud + union_genuine.sum()/n_genuine) / 2.0
        print(f"  [balanced_cov] {best_key[0][:40]:40s} | {best_key[1]:8s} "
              f"new_score={best_new_score:.4f} | total_score={total_score:.4f}")

        if total_score > best_score:
            best_score = total_score
            best_selection = selected.copy()
            best_union_fraud = union_fraud.copy()
            best_union_genuine = union_genuine.copy()

        # Stop if no model added improves total score
        if best_new_score <= 0:
            break

    if not best_selection:
        # Fallback: pick best single model by leaderboard F1
        best_key = max(candidate_info.items(), key=lambda kv: kv[1]["f1_leaderboard"])[0]
        best_selection = [best_key]
        best_union_fraud = coverage_fraud_masks[best_key]
        best_union_genuine = coverage_genuine_masks[best_key]
        best_score = (best_union_fraud.sum()/n_fraud + best_union_genuine.sum()/n_genuine) / 2.0

    coverage_fraud = best_union_fraud.sum() / n_fraud
    coverage_genuine = best_union_genuine.sum() / n_genuine

    print(f"  [balanced_cov] selected {len(best_selection)} models, "
          f"score={best_score:.4f}, fraud_cov={coverage_fraud:.4f}, genuine_cov={coverage_genuine:.4f}")

    return best_selection, best_score, coverage_fraud, coverage_genuine

In [10]:
# =====================================================================
#  Build Meta-Features (Validation & Test) from Selected Models
# =====================================================================
def build_meta_features(selected_keys, dataset_name, split="val"):
    """
    Combine the probability columns of selected models into a
    single DataFrame. Column name = '<variant>__<model>'.
    """
    meta_dict = {}
    for variant, model in selected_keys:
        proba = load_model_predictions(dataset_name, variant, model, split=split)
        col_name = f"{variant[:40]}__{model}"
        meta_dict[col_name] = proba
    return pd.DataFrame(meta_dict)

In [11]:
def evaluate_hybrid(dataset_name, selected_keys, meta_learners, y_val):
    """
    Train each meta-learner on validation meta-features and evaluate on test.
    Returns a list of metric dicts.
    """
    X_val_meta = build_meta_features(selected_keys, dataset_name, split="val")
    X_test_meta = build_meta_features(selected_keys, dataset_name, split="test")
    y_test = load_raw_labels(dataset_name, split="test")

    results = []
    for name, meta in meta_learners.items():
        # Train on validation
        meta.fit(X_val_meta, y_val)
        # Predict probabilities
        if hasattr(meta, "predict_proba"):
            test_proba = meta.predict_proba(X_test_meta)[:, 1]
        elif hasattr(meta, "decision_function"):
            from scipy.special import expit
            test_proba = expit(meta.decision_function(X_test_meta))
        else:
            raise ValueError(f"Meta-learner {name} has no probability/decision function")

        metrics = compute_metrics(y_test, test_proba)
        metrics["meta_learner"] = name
        metrics["dataset"] = dataset_name
        metrics["n_models"] = len(selected_keys)
        metrics["selected_models"] = " || ".join([f"{v[:30]}__{m}" for v, m in selected_keys])
        results.append(metrics)
        print(f"  [meta] {name:20s}  F1={metrics['f1']:.4f}  AUC-ROC={metrics['auc_roc']:.4f}  PR-AUC={metrics['auc_pr']:.4f}")
    return results, X_val_meta, X_test_meta, y_test

In [12]:
# =====================================================================
#  Main Execution: Run All Five Selection Strategies with Timing
# =====================================================================

all_results = {
    "fraud_coverage": [],   # Strategy 1
    "balanced": [],         # Strategy 2
    "exhaustive": [],       # Strategy 3 (old exhaustive, kept for comparison)
    "diverse_cv": [],       # Strategy 4 (heuristic beam + CV)
    "coverage_balanced": [] # Strategy 5 (balanced coverage greedy)
}

# Discover datasets
lb_files = list(Path(CONFIG["LEADERBOARD_DIR"]).glob("*_top10_L0_by_f1.csv"))
if not lb_files:
    raise FileNotFoundError("No leaderboard files found.")
dataset_names = [f.stem.replace("_top10_L0_by_f1", "") for f in lb_files]
print(f"[main] datasets found: {dataset_names}\n")

for ds in dataset_names:
    print(f"\n{'='*80}\nDataset: {ds}\n{'='*80}")
    try:
        y_val = load_raw_labels(ds, split="val")
        candidates_top10 = get_candidates(ds)                    # top10 for strategies 1,2,3
        candidates_all = get_best_per_model_all(ds)              # one per model type for strategies 4,5
        meta_learners = get_meta_learners(CONFIG["RANDOM_STATE"])

        # ---------- Strategy 1: Fraud‑Coverage Greedy ----------
        print("\n--- Strategy 1: Fraud‑Coverage Greedy ---")
        t0 = time.time()
        selected_keys_cov, coverage_cov = greedy_select_models(candidates_top10, y_val, ds)
        results_cov, X_val_cov, X_test_cov, y_test = evaluate_hybrid(
            ds, selected_keys_cov, meta_learners, y_val
        )
        strat1_time = time.time() - t0
        for r in results_cov:
            r["selection_strategy"] = "fraud_coverage"
            r["coverage"] = coverage_cov
            r["strategy_time_sec"] = strat1_time
        all_results["fraud_coverage"].extend(results_cov)

        # Save artefacts
        info = {"dataset": ds, "strategy": "fraud_coverage",
                "selected_models": [{"variant": v, "model": m} for v, m in selected_keys_cov],
                "fraud_coverage": float(coverage_cov),
                "n_models": int(len(selected_keys_cov)),
                "strategy_time_sec": float(strat1_time)}
        with open(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_models_fraud_coverage.json", "w") as f:
            json.dump(info, f, indent=2)
        X_val_cov.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_fraud_coverage_selected_val_meta.csv", index=False)
        X_test_cov.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_fraud_coverage_selected_test_meta.csv", index=False)

        # ---------- Strategy 2: Balanced Greedy ----------
        print("\n--- Strategy 2: Balanced Greedy (lambda=0.5) ---")
        t0 = time.time()
        lambda_fp = CONFIG.get("LAMBDA_FP", 0.5)
        selected_keys_bal, coverage_bal, false_alarms_bal = balanced_greedy_select_models(
            candidates_top10, y_val, ds, lambda_fp=lambda_fp
        )
        results_bal, X_val_bal, X_test_bal, _ = evaluate_hybrid(
            ds, selected_keys_bal, meta_learners, y_val
        )
        strat2_time = time.time() - t0
        for r in results_bal:
            r["selection_strategy"] = "balanced"
            r["coverage"] = coverage_bal
            r["false_alarms"] = false_alarms_bal
            r["strategy_time_sec"] = strat2_time
        all_results["balanced"].extend(results_bal)

        info = {"dataset": ds, "strategy": "balanced", "lambda_fp": lambda_fp,
                "selected_models": [{"variant": v, "model": m} for v, m in selected_keys_bal],
                "fraud_coverage": float(coverage_bal),
                "false_alarms": int(false_alarms_bal),
                "n_models": int(len(selected_keys_bal)),
                "strategy_time_sec": float(strat2_time)}
        with open(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_models_balanced.json", "w") as f:
            json.dump(info, f, indent=2)
        X_val_bal.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_balanced_selected_val_meta.csv", index=False)
        X_test_bal.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_balanced_selected_test_meta.csv", index=False)

        # ---------- Strategy 3: Exhaustive Subset Search (old) ----------
        print("\n--- Strategy 3: Exhaustive Subset Search (old) ---")
        t0 = time.time()
        selected_keys_exh, coverage_exh, val_f1_exh = exhaustive_subset_search(
            candidates_top10, y_val, ds,
            max_models=CONFIG["MAX_MODELS"],
            coverage_weight=1.0,
            size_penalty=0.01,
            patience=2
        )
        results_exh, X_val_exh, X_test_exh, _ = evaluate_hybrid(
            ds, selected_keys_exh, meta_learners, y_val
        )
        strat3_time = time.time() - t0
        for r in results_exh:
            r["selection_strategy"] = "exhaustive"
            r["coverage"] = coverage_exh
            r["val_f1_selection"] = val_f1_exh
            r["strategy_time_sec"] = strat3_time
        all_results["exhaustive"].extend(results_exh)

        info = {"dataset": ds, "strategy": "exhaustive",
                "selected_models": [{"variant": v, "model": m} for v, m in selected_keys_exh],
                "fraud_coverage": float(coverage_exh),
                "val_f1_selection": float(val_f1_exh),
                "n_models": int(len(selected_keys_exh)),
                "strategy_time_sec": float(strat3_time)}
        with open(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_models_exhaustive.json", "w") as f:
            json.dump(info, f, indent=2)
        X_val_exh.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_exhaustive_selected_val_meta.csv", index=False)
        X_test_exh.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_exhaustive_selected_test_meta.csv", index=False)

        # ---------- Strategy 4: Heuristic Beam + CV ----------
        print("\n--- Strategy 4: Heuristic Beam + CV F1 ---")
        t0 = time.time()
        selected_keys_div, best_meta_div, cv_f1_div, coverage_div = heuristic_cv_subset_search(
            candidates_all, y_val, ds,
            max_models=CONFIG["MAX_MODELS"],
            meta_learners=meta_learners,
            n_splits=3,
            patience=2
        )
        # Train only selected meta-learner on full validation
        final_meta = clone(meta_learners[best_meta_div])
        X_val_meta_div = build_meta_features(selected_keys_div, ds, split="val")
        X_test_meta_div = build_meta_features(selected_keys_div, ds, split="test")
        y_test = load_raw_labels(ds, split="test")
        final_meta.fit(X_val_meta_div, y_val)
        if hasattr(final_meta, "predict_proba"):
            test_proba_div = final_meta.predict_proba(X_test_meta_div)[:, 1]
        elif hasattr(final_meta, "decision_function"):
            from scipy.special import expit
            test_proba_div = expit(final_meta.decision_function(X_test_meta_div))
        else:
            raise ValueError("Selected meta-learner cannot produce probabilities")
        metrics_div = compute_metrics(y_test, test_proba_div)
        strat4_time = time.time() - t0
        metrics_div["meta_learner"] = best_meta_div
        metrics_div["selection_strategy"] = "diverse_cv"
        metrics_div["dataset"] = ds
        metrics_div["n_models"] = len(selected_keys_div)
        metrics_div["coverage"] = coverage_div
        metrics_div["cv_f1_selection"] = cv_f1_div
        metrics_div["selected_models"] = " || ".join([f"{v[:30]}__{m}" for v, m in selected_keys_div])
        metrics_div["strategy_time_sec"] = strat4_time
        all_results["diverse_cv"].append(metrics_div)

        info_div = {
            "dataset": ds, "strategy": "diverse_cv",
            "selected_models": [{"variant": v, "model": m} for v, m in selected_keys_div],
            "meta_learner": best_meta_div,
            "cv_f1_selection": float(cv_f1_div),
            "coverage": float(coverage_div),
            "n_models": int(len(selected_keys_div)),
            "strategy_time_sec": float(strat4_time)
        }
        with open(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_models_diverse_cv.json", "w") as f:
            json.dump(info_div, f, indent=2)
        X_val_meta_div.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_diverse_cv_selected_val_meta.csv", index=False)
        X_test_meta_div.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_diverse_cv_selected_test_meta.csv", index=False)

        # ---------- Strategy 5: Balanced Coverage Greedy ----------
        print("\n--- Strategy 5: Balanced Coverage Greedy ---")
        t0 = time.time()
        selected_keys_covbal, score_covbal, cov_fraud5, cov_gen5 = balanced_coverage_greedy(
            candidates_all, y_val, ds, max_models=CONFIG["MAX_MODELS"]
        )
        results_covbal, X_val_covbal, X_test_covbal, _ = evaluate_hybrid(
            ds, selected_keys_covbal, meta_learners, y_val
        )
        strat5_time = time.time() - t0
        for r in results_covbal:
            r["selection_strategy"] = "coverage_balanced"
            r["coverage"] = cov_fraud5
            r["genuine_coverage"] = cov_gen5
            r["coverage_score"] = score_covbal
            r["strategy_time_sec"] = strat5_time
        all_results["coverage_balanced"].extend(results_covbal)

        info_covbal = {
            "dataset": ds, "strategy": "coverage_balanced",
            "selected_models": [{"variant": v, "model": m} for v, m in selected_keys_covbal],
            "fraud_coverage": float(cov_fraud5),
            "genuine_coverage": float(cov_gen5),
            "score": float(score_covbal),
            "n_models": int(len(selected_keys_covbal)),
            "strategy_time_sec": float(strat5_time)
        }
        with open(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_models_coverage_balanced.json", "w") as f:
            json.dump(info_covbal, f, indent=2)
        X_val_covbal.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_coverage_balanced_selected_val_meta.csv", index=False)
        X_test_covbal.to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_coverage_balanced_selected_test_meta.csv", index=False)

        # Save true test labels once
        pd.Series(y_test, name="y_true").to_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_test_labels.csv", index=False)

    except Exception as e:
        print(f"[error] {ds}: {e}")

# Combine all results
all_results_df = pd.concat(
    [pd.DataFrame(all_results["fraud_coverage"]),
     pd.DataFrame(all_results["balanced"]),
     pd.DataFrame(all_results["exhaustive"]),
     pd.DataFrame(all_results["diverse_cv"]),
     pd.DataFrame(all_results["coverage_balanced"])],
    ignore_index=True
)
all_results_df.to_csv(Path(CONFIG["OUTPUT_DIR"]) / "optimized_hybrid_results.csv", index=False)
print(f"\n[main] all results saved -> {Path(CONFIG['OUTPUT_DIR']) / 'optimized_hybrid_results.csv'}")

[main] datasets found: ['Sparkov', 'EuropeanCard', 'IEEE-CIS']


Dataset: Sparkov
[candidates] Sparkov: 10 candidates
[candidates all] Sparkov: 14 unique model types

--- Strategy 1: Fraud‑Coverage Greedy ---
  [select] EditedNearestNeighbours--ANOVA_Percentil | cat      new fraud covered: 1345 | total covered: 1345/1901
  [select] EditedNearestNeighbours--ANOVA_Percentil | dl_cnn   new fraud covered:  79 | total covered: 1424/1901
  [select] TomekLinks--NoSelection_all--Sparkov--20 | cat      new fraud covered:  16 | total covered: 1440/1901
  [select] EditedNearestNeighbours--ANOVA_k30--Spar | cat      new fraud covered:   5 | total covered: 1445/1901
  [select] TomekLinks--ANOVA_k30--Sparkov--20260629 | cat      new fraud covered:   1 | total covered: 1446/1901
  [greedy] selected 5 models, fraud coverage = 0.7607 (1446/1901)
  [meta] LogisticRegression    F1=0.8089  AUC-ROC=0.9939  PR-AUC=0.8583
  [meta] RidgeClassifier       F1=0.8109  AUC-ROC=0.9938  PR-AUC=0.8585
  [meta] Random

In [13]:
# =====================================================================
#  Summary Table
# =====================================================================
summary_df = pd.read_csv(Path(CONFIG["OUTPUT_DIR"]) / "optimized_hybrid_results.csv")
print("\n[summary] Optimized Hybrid Results (Test Set):")
print(summary_df.to_string(index=False))

# Optionally show best meta-learner per dataset
best_per_dataset = summary_df.loc[summary_df.groupby("dataset")["f1"].idxmax()]
print("\n[summary] Best Meta-Learner per Dataset:")
print(best_per_dataset[["dataset", "meta_learner", "precision", "recall", "f1", "auc_roc", "auc_pr", "n_models"]].to_string(index=False))


[summary] Optimized Hybrid Results (Test Set):
 precision   recall       f1  auc_roc   auc_pr  n_pos  n_total       meta_learner      dataset  n_models                                                                                                                                                                                                           selected_models selection_strategy  coverage  strategy_time_sec  false_alarms  val_f1_selection  cv_f1_selection  genuine_coverage  coverage_score
  0.912477 0.726464 0.808915 0.993867 0.858275   1349   370479 LogisticRegression      Sparkov         5                        EditedNearestNeighbours--ANOVA__cat || EditedNearestNeighbours--ANOVA__dl_cnn || TomekLinks--NoSelection_all--S__cat || EditedNearestNeighbours--ANOVA__cat || TomekLinks--ANOVA_k30--Sparkov__cat     fraud_coverage  0.760652          32.230897           NaN               NaN              NaN               NaN             NaN
  0.920038 0.724981 0.810945 0.993789 0.8585

In [17]:
# =====================================================================
#  FINAL: Performance Comparison + Meta‑Learner Timing + Combined Table
# =====================================================================


# ------------------------- 1. Performance Comparison -------------------------
leaderboard_dir = Path(CONFIG["LEADERBOARD_DIR"])
optimized_results = pd.read_csv(Path(CONFIG["OUTPUT_DIR"]) / "optimized_hybrid_results.csv")

comparison_rows = []
datasets = sorted([f.stem.replace("_top10_L0_by_f1", "")
                   for f in leaderboard_dir.glob("*_top10_L0_by_f1.csv")])

for ds in datasets:
    l0 = pd.read_csv(leaderboard_dir / f"{ds}_top10_L0_by_f1.csv")
    best_l0 = l0.loc[l0["f1"].idxmax()]

    l1 = pd.read_csv(leaderboard_dir / f"{ds}_top3_L1_by_f1.csv")
    best_l1 = l1.loc[l1["f1"].idxmax()]

    opt = optimized_results[optimized_results["dataset"] == ds]
    if opt.empty:
        continue

    strategies = ["fraud_coverage", "balanced", "exhaustive", "diverse_cv", "coverage_balanced"]
    row = {"Dataset": ds,
           "Best Single F1": best_l0["f1"],
           "Best Previous Stack F1": best_l1["f1"]}
    for strat in strategies:
        sub = opt[opt["selection_strategy"] == strat]
        if sub.empty:
            continue
        best = sub.sort_values("f1", ascending=False).iloc[0]
        row[f"{strat}_F1"] = best["f1"]
        row[f"{strat}_Meta"] = best["meta_learner"]
        row[f"{strat}_N_Models"] = best["n_models"]
        row[f"{strat}_Time_s"] = round(best["strategy_time_sec"], 2)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print("\n[comparison] Summary of all approaches (Test Set, F1 primary):")
print(comparison_df.to_string(index=False))
comparison_df.to_csv(Path(CONFIG["OUTPUT_DIR"]) / "approach_comparison_with_time.csv", index=False)
print(f"[comparison] Saved -> {Path(CONFIG['OUTPUT_DIR']) / 'approach_comparison_with_time.csv'}")

# ------------------------- 2. Meta‑Learner Timing -------------------------
# Load results and prepare meta-learners
results_df = optimized_results  # reuse from above
strategy_prefix = {
    "fraud_coverage":   "fraud_coverage_selected",
    "balanced":         "balanced_selected",
    "exhaustive":       "exhaustive_selected",
    "diverse_cv":       "diverse_cv_selected",
    "coverage_balanced":"coverage_balanced_selected",
}
meta_learners = get_meta_learners(CONFIG["RANDOM_STATE"])

timing_records = []
for ds in results_df["dataset"].unique():
    y_val = load_raw_labels(ds, split="val")
    y_test = pd.read_csv(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_selected_test_labels.csv")["y_true"].values

    for strategy, prefix in strategy_prefix.items():
        sub = results_df[(results_df["dataset"] == ds) &
                         (results_df["selection_strategy"] == strategy)]
        if sub.empty:
            continue
        best_row = sub.loc[sub["f1"].idxmax()]
        meta_name = best_row["meta_learner"]
        n_models = best_row["n_models"]

        val_file = Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_{prefix}_val_meta.csv"
        test_file = Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_{prefix}_test_meta.csv"
        if not val_file.exists() or not test_file.exists():
            print(f"[timing] Missing files for {ds}/{strategy}, skipping")
            continue

        X_val_meta = pd.read_csv(val_file)
        X_test_meta = pd.read_csv(test_file)

        if meta_name not in meta_learners:
            print(f"[timing] Unknown meta-learner {meta_name}, skipping {ds}/{strategy}")
            continue
        meta = clone(meta_learners[meta_name])

        # Training time
        t0 = time.time()
        meta.fit(X_val_meta, y_val)
        train_time = time.time() - t0

        # Inference time
        t0 = time.time()
        if hasattr(meta, "predict_proba"):
            _ = meta.predict_proba(X_test_meta)
        elif hasattr(meta, "decision_function"):
            _ = meta.decision_function(X_test_meta)
        else:
            _ = meta.predict(X_test_meta)
        inference_time = time.time() - t0

        timing_records.append({
            "dataset": ds,
            "strategy": strategy,
            "meta_learner": meta_name,
            "n_models": n_models,
            "train_time_sec": round(train_time, 4),
            "inference_time_sec": round(inference_time, 6),
            "test_f1": best_row["f1"],
        })

timing_df = pd.DataFrame(timing_records)
print("\n[timing] Training & Inference Times for Final Meta‑Learners:")
print(timing_df.to_string(index=False))
timing_df.to_csv(Path(CONFIG["OUTPUT_DIR"]) / "meta_learner_timing.csv", index=False)
print(f"[timing] Saved -> {Path(CONFIG['OUTPUT_DIR']) / 'meta_learner_timing.csv'}")

# ------------------------- 3. Combined Timing & Performance -------------------------
rows = []
for ds in datasets:
    # Best single model
    l0 = pd.read_csv(leaderboard_dir / f"{ds}_top10_L0_by_f1.csv")
    best_l0 = l0.loc[l0["f1"].idxmax()]

    # Best previous stacking
    l1 = pd.read_csv(leaderboard_dir / f"{ds}_top3_L1_by_f1.csv")
    best_l1 = l1.loc[l1["f1"].idxmax()]

    row = {
        "Dataset": ds,
        "Single Model": best_l0["model"],
        "Single F1": best_l0["f1"],
        "Single Train Time (s)": np.nan,      # not available from original run
        "Single Inference Time (s)": np.nan,
        "Prev Stack F1": best_l1["f1"],
        "Prev Stack Train Time (s)": np.nan,  # not available from original run
        "Prev Stack Inference Time (s)": np.nan,
    }

    strategies = ["fraud_coverage", "balanced", "exhaustive", "diverse_cv", "coverage_balanced"]
    for strat in strategies:
        sub = results_df[(results_df["dataset"] == ds) &
                         (results_df["selection_strategy"] == strat)]
        if sub.empty:
            continue
        best = sub.loc[sub["f1"].idxmax()]
        row[f"{strat} F1"] = best["f1"]
        row[f"{strat} Meta"] = best["meta_learner"]
        row[f"{strat} N Models"] = best["n_models"]

        # Get timing from the timing_df we just built
        t_sub = timing_df[(timing_df["dataset"] == ds) &
                          (timing_df["strategy"] == strat) &
                          (timing_df["meta_learner"] == best["meta_learner"])]
        if not t_sub.empty:
            row[f"{strat} Train Time (s)"] = t_sub.iloc[0]["train_time_sec"]
            row[f"{strat} Inference Time (s)"] = t_sub.iloc[0]["inference_time_sec"]
        else:
            row[f"{strat} Train Time (s)"] = np.nan
            row[f"{strat} Inference Time (s)"] = np.nan

    rows.append(row)

combined_timing_df = pd.DataFrame(rows)
print("\n[timing] Combined performance & timing for all approaches:")
print(combined_timing_df.to_string(index=False))
combined_timing_df.to_csv(Path(CONFIG["OUTPUT_DIR"]) / "combined_timing_comparison.csv", index=False)
print(f"[timing] Saved -> {Path(CONFIG['OUTPUT_DIR']) / 'combined_timing_comparison.csv'}")


[comparison] Summary of all approaches (Test Set, F1 primary):
     Dataset  Best Single F1  Best Previous Stack F1  fraud_coverage_F1 fraud_coverage_Meta  fraud_coverage_N_Models  fraud_coverage_Time_s  balanced_F1   balanced_Meta  balanced_N_Models  balanced_Time_s  exhaustive_F1 exhaustive_Meta  exhaustive_N_Models  exhaustive_Time_s  diverse_cv_F1 diverse_cv_Meta  diverse_cv_N_Models  diverse_cv_Time_s  coverage_balanced_F1 coverage_balanced_Meta  coverage_balanced_N_Models  coverage_balanced_Time_s
EuropeanCard        0.844444                0.844444           0.850746           LinearSVC                        2                   3.90     0.854962 RidgeClassifier                  2             4.12       0.837209             MLP                    5               5.18       0.835821    RandomForest                    2             902.61              0.821705        RidgeClassifier                           3                      6.18
    IEEE-CIS        0.452511                

## Final Results Summary

| Dataset      | Best Single F1 | Best Previous Stack F1 | Fraud-Cov F1 | Fraud-Cov Meta | Fraud-Cov N | Fraud-Cov Time (s) | Balanced F1 | Balanced Meta | Balanced N | Balanced Time (s) | Exhaustive F1 | Exhaustive Meta | Exhaustive N | Exhaustive Time (s) | Diverse-CV F1 | Diverse-CV Meta | Diverse-CV N | Diverse-CV Time (s) | Coverage-Balanced F1 | Coverage-Balanced Meta | Coverage-Balanced N | Coverage-Balanced Time (s) |
|--------------|----------------|------------------------|--------------|----------------|-------------|--------------------|-------------|---------------|------------|-------------------|---------------|-----------------|--------------|---------------------|---------------|-----------------|--------------|---------------------|----------------------|------------------------|---------------------|---------------------------|
| EuropeanCard | 0.8444         | 0.8444                 | 0.8507       | LinearSVC      | 2           | 3.90               | 0.8550      | RidgeClassifier | 2          | 4.12              | 0.8372        | MLP             | 5            | 5.18                | 0.8358        | RandomForest    | 2            | 902.61              | 0.8217               | RidgeClassifier        | 3                   | 6.18                      |
| IEEE-CIS     | 0.4525         | 0.4648                 | 0.4556       | LightGBM       | 3           | 7.39               | 0.4523      | LightGBM        | 2          | 6.76              | 0.4558        | MLP             | 5            | 10.19               | 0.4582        | LightGBM        | 4            | 2157.08             | 0.4547               | MLP                    | 5                   | 10.40                     |
| Sparkov      | 0.8012         | 0.7794                 | 0.8174       | RandomForest   | 5           | 32.23              | 0.8209      | RandomForest    | 4          | 25.23             | 0.8184        | XGBoost         | 2            | 29.22               | 0.8252        | RandomForest    | 5            | 4014.84             | 0.8063               | RidgeClassifier        | 5                   | 27.15                     |

---

## Key Observations

1. **No single selection strategy dominates all datasets.**  
   - *EuropeanCard*: **Balanced Greedy** (0.8550) slightly outperforms the best single model and previous stacking, with only 2 base models and ~4 s selection time.  
   - *IEEE-CIS*: **Previous blind stacking** remains the best (0.4648), followed closely by the new strategies; differences are small.  
   - *Sparkov*: **Diverse-CV (Strategy 4)** achieves the highest F1 (0.8252), clearly beating the best single model (0.8012) and blind stacking (0.7794), but at a huge computational cost (~67 min).

2. **Blind stacking is not always beneficial.**  
   On Sparkov, the previous blind stacking underperformed the best single model by ~0.022 F1. This confirms that careful model selection is necessary; simply combining many models can hurt if they are redundant or poorly chosen.

3. **Computational cost vs. performance trade‑off is critical.**  
   - Fast greedy heuristics (**Fraud-Cov**, **Balanced**, **Coverage-Balanced**) achieve near‑optimal results in seconds to tens of seconds.  
   - **Diverse-CV** can yield slightly higher F1 on some datasets but consumes 15–67 minutes, making it impractical for many applications.  
   - This trade‑off should be highlighted as a major design consideration for real‑world fraud detection systems.

4. **Meta‑learner choice varies and simple models often suffice.**  
   - **RidgeClassifier**, **LinearSVC**, and **Logistic Regression** frequently appear among the best meta‑learners, showing that a well‑selected set of base probabilities can be effectively combined by a linear model.  
   - **RandomForest** performs well on Sparkov; **LightGBM** is competitive on IEEE-CIS.  
   - The meta‑learner should therefore be tuned per dataset or chosen via validation, as done in Strategy 4.

5. **Candidate diversity improves ensemble quality.**  
   - Strategy 4 used one best variant per model type (from all trained models) instead of the top‑10 overall. This helped on Sparkov (F1 0.825 vs 0.817–0.821 for top‑10-based strategies).  
   - However, on EuropeanCard the diverse pool was smaller and performed slightly worse, indicating that diversity alone is not sufficient; dataset characteristics also matter.

6. **Validation‑only selection keeps results realistic.**  
   - All model selection and hyperparameter tuning were performed on the validation set. The test set was used only once for final evaluation.  
   - This ensures that reported F1 scores are not optimistic and can be trusted for generalization claims.

---